# Compare Aβ42 WT and MUTANT (pentamers)

## 0. Loading and preparation

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align, distances
from MDAnalysis.analysis.rms import RMSF
from MDAnalysis.analysis.dssp import DSSP
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
from MDAnalysis.analysis import pca as mda_pca
from scipy import stats

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 10

# ── Colors ──────────────────────────────────────────────────────────
C_WT = "#2E86AB"  # blue — WT
C_MUT = "#6A0572"  # MUTANT

# ── File paths ──────────────────────────────
GRO_WT = "../md_wt/md.gro"
XTC_WT = "../md_wt/md_clean.xtc"

GRO_MUT = "./md.gro"
XTC_MUT = "./md_clean.xtc"

# ── Mutation positions ────────────────────────────────────────────────

MUT_POS = int(input())
MUT_POSITIONS = [
    MUT_POS,
    MUT_POS + 42,
    MUT_POS + 42 * 2,
    MUT_POS + 42 * 3,
    MUT_POS + 42 * 4,
]
MUT_LABEL = input()

# ── Loading ───────────────────────────────────────────────────────
print("Loading WT  ...")
u_wt = mda.Universe(GRO_WT, XTC_WT)
ref_wt = mda.Universe(GRO_WT, XTC_WT)

print("Loading MUTANT ...")
u_mut = mda.Universe(GRO_MUT, XTC_MUT)
ref_mut = mda.Universe(GRO_MUT, XTC_MUT)

for label, u in [("WT", u_wt), (MUT_LABEL, u_mut)]:
    print(
        f"\n{label}: {u.atoms.n_atoms} atoms | "
        f"{u.atoms.n_residues} residues | "
        f"{u.trajectory.n_frames} frames | "
        f"{u.trajectory.n_frames * u.trajectory.dt / 1000:.1f} ns"
    )

## 1. RMSD

In [ ]:
print("Calculating RMSD...")


def get_rmsd(u, ref):
    align.AlignTraj(u, ref, select="protein and name CA", in_memory=False).run()
    R = rms.RMSD(u, ref, select="protein and name CA").run()
    time_ns = R.results.rmsd[:, 1] / 1000
    rmsd_ca = R.results.rmsd[:, 2]
    return time_ns, rmsd_ca


t_wt, rmsd_wt = get_rmsd(u_wt, ref_wt)
t_mut, rmsd_mut = get_rmsd(u_mut, ref_mut)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Time Series ─────────────────────────────────────────────────
axes[0].plot(t_wt, rmsd_wt, color=C_WT, lw=0.8, alpha=0.85, label="WT")
axes[0].plot(t_mut, rmsd_mut, color=C_MUT, lw=0.8, alpha=0.85, label=MUT_LABEL)
axes[0].set_xlabel("Time (ns)")
axes[0].set_ylabel("RMSD (Å)")
axes[0].set_title("Cα RMSD over time", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ── Distribution ──────────────────────────────────────────────────
axes[1].hist(
    rmsd_wt,
    bins=50,
    density=True,
    color=C_WT,
    alpha=0.55,
    label="WT",
    edgecolor="white",
)
axes[1].hist(
    rmsd_mut,
    bins=50,
    density=True,
    color=C_MUT,
    alpha=0.55,
    label=MUT_LABEL,
    edgecolor="white",
)
for val, col in [(np.mean(rmsd_wt), C_WT), (np.mean(rmsd_mut), C_MUT)]:
    axes[1].axvline(val, color=col, lw=2, linestyle="--")
axes[1].set_xlabel("RMSD (Å)")
axes[1].set_ylabel("Probability density")
axes[1].set_title("RMSD distribution", fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.suptitle(
    f"Cα RMSD: WT vs {MUT_LABEL} pentamer", fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(f"01_rmsd_wt_vs_{MUT_LABEL.lower()}.png", dpi=300, bbox_inches="tight")
plt.show()

t_stat, p_val = stats.ttest_ind(rmsd_wt, rmsd_mut)
print("\nRMSD statistics:")
print(f"  WT:          {np.mean(rmsd_wt):.2f} ± {np.std(rmsd_wt):.2f} Å")
print(f"  {MUT_LABEL}: {np.mean(rmsd_mut):.2f} ± {np.std(rmsd_mut):.2f} Å")
print(f"  Δ mean:      {np.mean(rmsd_mut) - np.mean(rmsd_wt):+.2f} Å")
print(
    f"  t-test:      t={t_stat:.2f}, p={p_val:.4f} {'***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'}"
)

## 2. RMSF

In [ ]:
print("Calculating RMSF...")

N_MON = 5
N_RES = 42
mon_resids = np.arange(1, N_RES + 1)


def get_rmsf_per_monomer(u, ref):
    align.AlignTraj(u, ref, select="protein and name CA", in_memory=True).run()
    ca_all = u.select_atoms("protein and name CA")

    profiles = []
    for i in range(N_MON):
        ca_mon = ca_all[i * N_RES : (i + 1) * N_RES]
        rmsf_mon = RMSF(ca_mon).run().results.rmsf
        profiles.append(rmsf_mon)

    profiles = np.array(profiles)  # (5, 42)
    mean_rmsf = profiles.mean(axis=0)  # (42,)
    sem_rmsf = profiles.std(axis=0) / np.sqrt(N_MON)
    return mon_resids, mean_rmsf, sem_rmsf, profiles


resids, rmsf_wt, sem_wt, prof_wt = get_rmsf_per_monomer(u_wt, ref_wt)
resids, rmsf_mut, sem_mut, prof_mut = get_rmsf_per_monomer(u_mut, ref_mut)

# Mutation position within monomer
mut_res_in_mon = MUT_POSITIONS[0] % N_RES
if mut_res_in_mon == 0:
    mut_res_in_mon = N_RES

# ── Picture 1: profiles ± SEM ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

for rmsf, sem, col, lbl in [
    (rmsf_wt, sem_wt, C_WT, "WT"),
    (rmsf_mut, sem_mut, C_MUT, MUT_LABEL),
]:
    ax.plot(resids, rmsf, color=col, lw=1.8, label=lbl)
    ax.fill_between(resids, rmsf - sem, rmsf + sem, color=col, alpha=0.18)
    ax.axhline(np.mean(rmsf), color=col, lw=1.2, ls="--", alpha=0.55)

ax.axvline(
    mut_res_in_mon,
    color="green",
    lw=1.5,
    ls=":",
    label=f"Mutation site (res {mut_res_in_mon})",
)

ax.set_xlabel("Residue number (within monomer)", fontsize=11)
ax.set_ylabel("RMSF (Å)", fontsize=11)
ax.set_xticks(range(1, N_RES + 1, 5))
ax.set_xlim(1, N_RES)
ax.set_title(
    f"Per-residue RMSF: WT vs {MUT_LABEL}  (mean ± SEM over {N_MON} monomers)",
    fontweight="bold",
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"02_rmsf_wt_vs_{MUT_LABEL.lower()}.png", dpi=300, bbox_inches="tight")
plt.show()

# ── Picture 2: ΔRMSF ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))

delta = rmsf_mut - rmsf_wt
ax.bar(
    resids,
    delta,
    color=[C_MUT if d > 0 else C_WT for d in delta],
    alpha=0.75,
    width=0.85,
)
ax.axhline(0, color="black", lw=1)
ax.axvline(
    mut_res_in_mon,
    color="green",
    lw=1.5,
    ls=":",
    label=f"Mutation site (res {mut_res_in_mon})",
)

ax.set_xlabel("Residue number (within monomer)", fontsize=11)
ax.set_ylabel("ΔRMSF (Å)  [mut − WT]", fontsize=11)
ax.set_xticks(range(1, N_RES + 1, 5))
ax.set_xlim(0.5, N_RES + 0.5)
ax.set_title(
    f"ΔRMSF: {MUT_LABEL} − WT  (positive = more flexible in mutant)", fontweight="bold"
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(
    f"02b_delta_rmsf_wt_vs_{MUT_LABEL.lower()}.png", dpi=300, bbox_inches="tight"
)
plt.show()

## 3. Radius of gyration

In [ ]:
print("Calculating Rg...")


def get_rg(u):
    prot = u.select_atoms("protein")
    rg, t = [], []
    for ts in u.trajectory:
        rg.append(prot.radius_of_gyration() / 10)  # Å → nm
        t.append(ts.time / 1000)
    return np.array(t), np.array(rg)


t_wt, rg_wt = get_rg(u_wt)
t_mut, rg_mut = get_rg(u_mut)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(t_wt, rg_wt, color=C_WT, lw=0.7, alpha=0.85, label="WT")
axes[0].plot(t_mut, rg_mut, color=C_MUT, lw=0.7, alpha=0.85, label=MUT_LABEL)
axes[0].axhline(np.mean(rg_wt), color=C_WT, lw=1.5, ls="--")
axes[0].axhline(np.mean(rg_mut), color=C_MUT, lw=1.5, ls="--")
axes[0].set_xlabel("Time (ns)")
axes[0].set_ylabel("Rg (nm)")
axes[0].set_title("Radius of gyration over time", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(
    rg_wt, bins=50, density=True, color=C_WT, alpha=0.55, label="WT", edgecolor="white"
)
axes[1].hist(
    rg_mut,
    bins=50,
    density=True,
    color=C_MUT,
    alpha=0.55,
    label=MUT_LABEL,
    edgecolor="white",
)
axes[1].axvline(np.mean(rg_wt), color=C_WT, lw=2, ls="--")
axes[1].axvline(np.mean(rg_mut), color=C_MUT, lw=2, ls="--")
axes[1].set_xlabel("Rg (nm)")
axes[1].set_ylabel("Probability density")
axes[1].set_title("Rg distribution", fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.suptitle(
    f"Radius of gyration: WT vs {MUT_LABEL} pentamer",
    fontsize=13,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.savefig(f"03_rg_wt_vs_{MUT_LABEL.lower()}.png", dpi=300, bbox_inches="tight")
plt.show()

t_stat, p_val = stats.ttest_ind(rg_wt, rg_mut)
print("\nRg Statistics:")
print(f"  WT:          {np.mean(rg_wt):.3f} ± {np.std(rg_wt):.3f} nm")
print(f"  {MUT_LABEL}: {np.mean(rg_mut):.3f} ± {np.std(rg_mut):.3f} nm")
print(f"  Δ mean:      {np.mean(rg_mut) - np.mean(rg_wt):+.4f} nm")
print(
    f"  t-test:      t={t_stat:.2f}, p={p_val:.4f} {'***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'}"
)

## 4. DSSP-analysis (Secondary structure)

In [ ]:
print("Calculating DSSP...")

N_MON = 5
N_RES = 42
mon_resids = np.arange(1, N_RES + 1)


def get_dssp_per_monomer(u):
    d = DSSP(u).run().results.dssp_ndarray  # (n_frames, 210, 3)

    h_profiles, s_profiles, c_profiles = [], [], []
    for i in range(N_MON):
        sl = slice(i * N_RES, (i + 1) * N_RES)
        h_profiles.append(d[:, sl, 1].mean(axis=0))  # mean frames
        s_profiles.append(d[:, sl, 2].mean(axis=0))
        c_profiles.append(d[:, sl, 0].mean(axis=0))

    h = np.array(h_profiles).mean(axis=0)  # mean monomers
    s = np.array(s_profiles).mean(axis=0)
    c = np.array(c_profiles).mean(axis=0)
    return mon_resids, h, s, c


resids_wt, h_wt, s_wt, c_wt = get_dssp_per_monomer(u_wt)
resids_mut, h_mut, s_mut, c_mut = get_dssp_per_monomer(u_mut)

mut_res_in_mon = MUT_POSITIONS[0] % N_RES
if mut_res_in_mon == 0:
    mut_res_in_mon = N_RES

# ── Picture 1: WT and MUTANT bars ────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(11, 9), sharex=True)

for ax, resids, h, s, c, title in [
    (axes[0], resids_wt, h_wt, s_wt, c_wt, "WT"),
    (axes[1], resids_mut, h_mut, s_mut, c_mut, MUT_LABEL),
]:
    ax.bar(resids, h, width=0.85, color="#E63946", alpha=0.8, label="Helix")
    ax.bar(resids, s, width=0.85, bottom=h, color="#457B9D", alpha=0.8, label="Sheet")
    ax.bar(
        resids, c, width=0.85, bottom=h + s, color="#CCCCCC", alpha=0.6, label="Coil"
    )
    ax.axvline(
        mut_res_in_mon,
        color="green",
        lw=1.5,
        ls=":",
        label=f"Mutation site (res {mut_res_in_mon})",
    )
    ax.set_ylabel("Fraction")
    ax.set_ylim(0, 1)
    ax.set_xlim(0.5, N_RES + 0.5)
    ax.set_xticks(range(1, N_RES + 1, 5))
    ax.set_title(f"Secondary structure propensity — {title}", fontweight="bold")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, alpha=0.2, axis="y")

axes[1].set_xlabel("Residue number (within monomer)", fontsize=11)
plt.tight_layout()
plt.savefig(f"04_dssp_wt_vs_{MUT_LABEL.lower()}.png", dpi=300, bbox_inches="tight")
plt.show()

# ── Picture 2: ΔDSSP ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

for ax, delta, ylabel, color in [
    (axes[0], h_mut - h_wt, "ΔHelix fraction", "#E63946"),
    (axes[1], s_mut - s_wt, "ΔSheet fraction", "#457B9D"),
]:
    ax.bar(
        mon_resids,
        delta,
        color=[color if d > 0 else C_WT for d in delta],
        width=0.85,
        alpha=0.8,
    )
    ax.axhline(0, color="black", lw=1)
    ax.axvline(
        mut_res_in_mon,
        color="green",
        lw=1.5,
        ls=":",
        label=f"Mutation site (res {mut_res_in_mon})",
    )
    ax.set_ylabel(f"{ylabel}  [{MUT_LABEL} − WT]")
    ax.set_xlim(0.5, N_RES + 0.5)
    ax.set_xticks(range(1, N_RES + 1, 5))
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2, axis="y")

axes[1].set_xlabel("Residue number (within monomer)", fontsize=11)
plt.suptitle(f"ΔDSSP: {MUT_LABEL} − WT", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(
    f"04b_delta_dssp_wt_vs_{MUT_LABEL.lower()}.png", dpi=300, bbox_inches="tight"
)
plt.show()

print("\nMean secondary structure:")
for name, h, s, c in [("WT", h_wt, s_wt, c_wt), (MUT_LABEL, h_mut, s_mut, c_mut)]:
    print(
        f"  {name}: helix {np.mean(h) * 100:.1f}%  sheet {np.mean(s) * 100:.1f}%  coil {np.mean(c) * 100:.1f}%"
    )

## 5. Contact maps

In [ ]:
print("Calculating contact maps (every 10th frame)...")

N_MON = 5
N_RES = 42
mon_resids = np.arange(1, N_RES + 1)


def get_avg_dist_per_monomer(u):
    ca = u.select_atoms("protein and name CA")
    n = len(ca)  # 210
    acc = np.zeros((n, n))
    cnt = 0
    for ts in u.trajectory[::10]:
        flat = distances.self_distance_array(ca.positions)
        sq = np.zeros((n, n))
        sq[np.triu_indices(n, k=1)] = flat
        sq += sq.T
        acc += sq
        cnt += 1
    dm_full = acc / cnt
    dm_mon = np.zeros((N_RES, N_RES))
    count = 0
    for i in range(N_MON):
        for j in range(N_MON):
            ri = slice(i * N_RES, (i + 1) * N_RES)
            rj = slice(j * N_RES, (j + 1) * N_RES)
            dm_mon += dm_full[ri, rj]
            count += 1
    return mon_resids, dm_mon / count


resids, dm_wt = get_avg_dist_per_monomer(u_wt)
resids, dm_mut = get_avg_dist_per_monomer(u_mut)

extent = [1, N_RES, 1, N_RES]
vmin = min(dm_wt.min(), dm_mut.min())
vmax = max(np.percentile(dm_wt, 95), np.percentile(dm_mut, 95))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, dm, title in [
    (axes[0], dm_wt, "WT"),
    (axes[1], dm_mut, MUT_LABEL),
]:
    im = ax.imshow(
        dm,
        cmap="YlOrRd_r",
        origin="lower",
        extent=extent,
        aspect="auto",
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(f"Avg Cα–Cα distance — {title}", fontweight="bold")
    ax.set_xlabel("Residue (within monomer)")
    ax.set_ylabel("Residue (within monomer)")
    ax.set_xticks(range(1, N_RES + 1, 5))
    ax.set_yticks(range(1, N_RES + 1, 5))
    plt.colorbar(im, ax=ax, label="Distance (Å)")

# Distance map
diff = dm_mut - dm_wt
lim = np.percentile(np.abs(diff), 97)
im2 = axes[2].imshow(
    diff, cmap="RdBu", origin="lower", extent=extent, aspect="auto", vmin=-lim, vmax=lim
)
axes[2].set_title(f"Δ distance: {MUT_LABEL} − WT", fontweight="bold")
axes[2].set_xlabel("Residue (within monomer)")
axes[2].set_ylabel("Residue (within monomer)")
axes[2].set_xticks(range(1, N_RES + 1, 5))
axes[2].set_yticks(range(1, N_RES + 1, 5))
plt.colorbar(im2, ax=axes[2], label="Δ Distance (Å)")

plt.tight_layout()
plt.savefig(
    f"05_contact_maps_wt_vs_{MUT_LABEL.lower()}.png", dpi=300, bbox_inches="tight"
)
plt.show()

## 6. H-bonds

In [ ]:
print("Calculating backbone hydrogen bonds...")


def get_hbonds(u):
    hb = HydrogenBondAnalysis(
        universe=u,
        donors_sel="protein and name N",
        hydrogens_sel="protein and name H HN",
        acceptors_sel="protein and name O",
        d_a_cutoff=3.5,
        d_h_a_angle_cutoff=120,
    )
    hb.run()
    frames = np.unique(hb.results.hbonds[:, 0])
    counts = np.array([np.sum(hb.results.hbonds[:, 0] == f) for f in frames])
    times = frames * u.trajectory.dt / 1000
    return times, counts


t_hb_wt, hb_wt = get_hbonds(u_wt)
t_hb_mut, hb_mut = get_hbonds(u_mut)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, t, hb, col, lbl in [
    (axes[0], t_hb_wt, hb_wt, C_WT, "WT"),
    (axes[0], t_hb_mut, hb_mut, C_MUT, MUT_LABEL),
]:
    ax.plot(t, hb, color=col, lw=0.7, alpha=0.8, label=lbl)
    ax.axhline(np.mean(hb), color=col, lw=1.5, ls="--")
axes[0].set_xlabel("Time (ns)")
axes[0].set_ylabel("H-bonds (backbone)")
axes[0].set_title("Backbone H-bonds over time", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(
    hb_wt, bins=30, density=True, color=C_WT, alpha=0.55, label="WT", edgecolor="white"
)
axes[1].hist(
    hb_mut,
    bins=30,
    density=True,
    color=C_MUT,
    alpha=0.55,
    label=MUT_LABEL,
    edgecolor="white",
)
axes[1].set_xlabel("Number of H-bonds")
axes[1].set_ylabel("Probability density")
axes[1].set_title("H-bond count distribution", fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis="y")

plt.suptitle(
    f"Backbone H-bonds: WT vs {MUT_LABEL}", fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig(f"06_hbonds_wt_vs_{MUT_LABEL.lower()}.png", dpi=300, bbox_inches="tight")
plt.show()

t_stat, p_val = stats.ttest_ind(hb_wt, hb_mut)
print("\nH-bonds Statistics:")
print(f"  WT:          {np.mean(hb_wt):.1f} ± {np.std(hb_wt):.1f}")
print(f"  {MUT_LABEL}: {np.mean(hb_mut):.1f} ± {np.std(hb_mut):.1f}")
print(f"  Δ mean:      {np.mean(hb_mut) - np.mean(hb_wt):+.1f}")
print(
    f"  t-test:      t={t_stat:.2f}, p={p_val:.4f} {'***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'}"
)

## 7. PCA

In [ ]:
print("Running PCA...")


def get_pca(u):
    pc = mda_pca.PCA(u, select="protein and name CA", align=True).run()
    ca = u.select_atoms("protein and name CA")
    tr = pc.transform(ca, n_components=3)
    return pc, tr


pc_wt, tr_wt = get_pca(u_wt)
pc_mut, tr_mut = get_pca(u_mut)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulated variance
axes[0].plot(
    range(1, 21),
    pc_wt.results.cumulated_variance[:20] * 100,
    "o-",
    color=C_WT,
    lw=2,
    ms=5,
    label="WT",
)
axes[0].plot(
    range(1, 21),
    pc_mut.results.cumulated_variance[:20] * 100,
    "s-",
    color=C_MUT,
    lw=2,
    ms=5,
    label=MUT_LABEL,
)
axes[0].axhline(80, color="red", ls="--", lw=1.5, label="80%")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Cumulated variance (%)")
axes[0].set_title("Variance explained by PCs", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# PC1 vs PC2 projection
sc1 = axes[1].scatter(
    tr_wt[:, 0],
    tr_wt[:, 1],
    c=np.linspace(0, 1, len(tr_wt)),
    cmap="Blues",
    s=4,
    alpha=0.5,
    label="WT",
)
sc2 = axes[1].scatter(
    tr_mut[:, 0],
    tr_mut[:, 1],
    c=np.linspace(0, 1, len(tr_mut)),
    cmap="Purples",
    s=4,
    alpha=0.5,
    label=MUT_LABEL,
)
axes[1].set_xlabel("PC1 (Å)")
axes[1].set_ylabel("PC2 (Å)")
axes[1].set_title("PC1 vs PC2 projection", fontweight="bold")
p1 = mpatches.Patch(color=C_WT, label="WT")
p2 = mpatches.Patch(color=C_MUT, label=MUT_LABEL)
axes[1].legend(handles=[p1, p2])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"07_pca_wt_vs_{MUT_LABEL.lower()}.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nPCA дисперсия (PC1+PC2+PC3):")
print(f"  WT:          {pc_wt.results.cumulated_variance[2] * 100:.1f}%")
print(f"  {MUT_LABEL}: {pc_mut.results.cumulated_variance[2] * 100:.1f}%")